# Bisection Method of Finding Root

# 1. Introduction:

- The bisection method (also known as the interval halving method or binary-search method) is one of the most basic and reliable numerical techniques for finding a root of an equation $f(x)=0$. 
- It is based on the Intermediate Value Theorem -  if $f$ is a continuous function whose domain contains the interval $[a, b]$ and $s$ is a number such that $f(a) < s < f(b)$, then there exists some $x \in (a,b)$ such that $f(x)=s$.

# 2. Condition:

- Before applying the method, the root must be "bracketed" within a specific interval $(a, b)$, which means there should be only one solution in the interval $(a, b)$. 

- To successfully isolate a root, the method requires another two conditions:
    + The function $f(x)$ must be continuous on the interval $[a, b]$.
    + The function values at the endpoints must have opposite signs, meaning $f(a) \ldotp f(b) < 0$. This guarantees that the function crosses the x-axis at least once within this interval.

# 3. Algorithm:

- The procedure systematically narrows down the interval containing the root by halving it at each step: Set $a_1 = a$ and $b_1 = b$.
    + Find the midpoint $x_n$ of the current interval: $x_n= \dfrac{a_n+b_n}{2}$ (to avoid overflow, should use $x_n = a_n + \dfrac{b_n - a_n}{2}$).
    + Evaluate the function at the midpoint, $f(x_n)$.
    + Determine the new interval:
        + If $f(x_n) = 0$, then $x_n$ is the exact root, and the algorithm stops.
        + If $f(a_n) \ldotp f(x_n) < 0$, the root lies in the left subinterval. Next upper bound $b_{n+1}$ is replaced by $x_n$, while next lower bound $a_{n+1}$ is kept unchanged from $a_n$.
        + If $f(a_n) \ldotp f(x_n) > 0$, the root lies in the right subinterval. Next lower bound $a_{n+1}$ is replaced by $x_n$, while next upper bound $b_{n+1}$ is kept unchanged from $b_n$.
    + Repeat the process for the new subinterval until a stopping criterion is met. Return $x_n$.

- Stopping Criteria: Because the method approaches the root infinitely, it must be terminated based on a predefined error tolerance. Common stopping criteria include:
    + number of iteration steps pre-defined using priori error estimation,
    + or if the function value is extremely close to zero: $|f(x_n)| < \varepsilon$,
    + or if the relative error between iterations has become sufficiently small: $|x_n - x_{n-1}| \leq \epsilon$,
    + or if the absolute error between iterations falls below a specific percentage: $\dfrac{|x_n - x_{n-1}|}{|x_n|} \leq \eta $.
    
    Without additional information about function $f$ or solution $x^*$, the best practice is to use absolute error as stopping condition due to it being closest to testing relative error. There is counter-example for using some other posteriori error estimate like function value or relative error.

# 4. Judgement:

- Convergence and Error Estimation:
    + Linear Convergence: The bisection method converges linearly due to the error is halved after each iteration
    + Error Bound: The absolute error after n bisections is strictly bounded by the formula: $∣x_n - x^*| \leq \dfrac{b_n - a_n}{2} = \dfrac{b-a}{2^n}$.
    + Predictable Iterations: You can calculate exactly how many iterations $n$ are required to achieve a desired accuracy $\epsilon$ using the formula: $n=\log_2 \left( \dfrac{b-a}{\epsilon} \right)$.
​
- Advantages:
    + The greatest advantage of the bisection method is its guaranteed reliability; once a root is bracketed, the method will always converge to it. 
    + It is also highly robust, even at the limits of computer machine precision.

- Disadvantage:
    + Its primary drawback is that it is relatively slow to converge compared to other techniques like Newton's Method or the Secant Method. 
    + Additionally, the bisection method cannot detect double roots (where the curve touches the x-axis but does not cross it) because there is no sign change.


# 5. Implementation

In [4]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 9)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

sign = lambda x: 1 if x > 0 else (-1 if x < 0 else 0)

## 5.1. Priori Estimation with number of iterations

- Error Estimation at each iteration: $\Delta {x_n} = \dfrac{b-a}{2^n}$

- If no error tolerance is given, you have to add rounded error to the final $\Delta{x_n}$.


In [7]:
def bisection_iteration_v1(f, a, b, max_iterations, decimal_places, type):
    # Error function
    if (f(a) * f(b) >= 0):
        print("You have not assumed right a and b\n")
        return

    # Implementing Bisection Method
    a_n = a; b_n = b; x_n = 0; delta = 0;

    results = []; diff = b - a; temp_2 = 2;
    for i in range(1, max_iterations + 1):
        # Find next value of x
        x_n = a_n + (b_n - a_n) / 2.0
        delta = diff / temp_2
        temp_2 *= 2

        results.append({
            'n': i,
            'a_n': a_n,
            'b_n': b_n,
            'x_n': x_n,
            'f(a_n)': f(a_n),
            'f(b_n)': f(b_n),
            'f(x_n)': f(x_n),
            'delta=(b-a)/2^(n+1)': delta
        })

        # Prepare for next iteration
        if f(x_n) == 0:
            break
        elif (f(a_n) * f(x_n) < 0):
            b_n = x_n
        elif (f(a_n) * f(x_n) > 0):
            a_n = x_n

    # Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))

    if decimal_places == None:
        print(f"The value of root is: {x_n}")
    else:
        if type == "no":
            total_delta = delta + 0.5 * 10**(-decimal_places) #must calculate roundoff error
        elif type == "yes":
            total_delta = delta #error tolerance was given so no need for roundoff error
        print(f"The value of root with {decimal_places} decimal point is: {round(x_n, decimal_places)}")
        print(f"Relative error is: {total_delta}")

In [8]:
f = lambda x: np.e**x - np.cos(2*x)

a = -3
b = -2

max_iterations = 20
decimal_places = 5

bisection_iteration_v1(f, a, b, max_iterations, decimal_places, type="no")

 n          a_n          b_n          x_n       f(a_n)      f(b_n)       f(x_n)  delta=(b-a)/2^(n+1)
 1 -3.000000000 -2.000000000 -2.500000000 -0.910383218 0.788978904 -0.201577187          0.500000000
 2 -2.500000000 -2.000000000 -2.250000000 -0.201577187 0.788978904  0.316195024          0.250000000
 3 -2.500000000 -2.250000000 -2.375000000 -0.201577187 0.316195024  0.055412336          0.125000000
 4 -2.500000000 -2.375000000 -2.437500000 -0.201577187 0.055412336 -0.074516304          0.062500000
 5 -2.437500000 -2.375000000 -2.406250000 -0.074516304 0.055412336 -0.009791147          0.031250000
 6 -2.406250000 -2.375000000 -2.390625000 -0.009791147 0.055412336  0.022765822          0.015625000
 7 -2.406250000 -2.390625000 -2.398437500 -0.009791147 0.022765822  0.006474264          0.007812500
 8 -2.406250000 -2.398437500 -2.402343750 -0.009791147 0.006474264 -0.001661945          0.003906250
 9 -2.402343750 -2.398437500 -2.400390625 -0.001661945 0.006474264  0.002405313          0.

- You can calculate number of iterations $n$ needed with given error tolerance $\epsilon$ using: $n=\log_2 \left( \dfrac{b-a}{\epsilon} \right)$, taking the smallest integer larger or equal that result.

In [9]:
f = lambda x: np.log(x) - 1 #approximate e

a = 2
b = 3

eps = 0.5 * pow(10, -7) #accurate to 7 trusted decimal places
max_iterations = int(np.ceil(np.log2((b-a)/eps)))
decimal_places = 7

bisection_iteration_v1(f, a, b, max_iterations, decimal_places, type="yes")

 n         a_n         b_n         x_n       f(a_n)      f(b_n)       f(x_n)  delta=(b-a)/2^(n+1)
 1 2.000000000 3.000000000 2.500000000 -0.306852819 0.098612289 -0.083709268          0.500000000
 2 2.500000000 3.000000000 2.750000000 -0.083709268 0.098612289  0.011600912          0.250000000
 3 2.500000000 2.750000000 2.625000000 -0.083709268 0.011600912 -0.034919104          0.125000000
 4 2.625000000 2.750000000 2.687500000 -0.034919104 0.011600912 -0.011388607          0.062500000
 5 2.687500000 2.750000000 2.718750000 -0.011388607 0.011600912  0.000172216          0.031250000
 6 2.687500000 2.718750000 2.703125000 -0.011388607 0.000172216 -0.005591489          0.015625000
 7 2.703125000 2.718750000 2.710937500 -0.005591489 0.000172216 -0.002705484          0.007812500
 8 2.710937500 2.718750000 2.714843750 -0.002705484 0.000172216 -0.001265599          0.003906250
 9 2.714843750 2.718750000 2.716796875 -0.001265599 0.000172216 -0.000546433          0.001953125
10 2.716796875 2.718

In [10]:
f = lambda x: np.tan(x/4) - 1 #approximate pi

a = 3
b = 3.2

eps = 0.5 * pow(10, -7) #accurate to 7 trusted decimal places
max_iterations = int(np.ceil(np.log2((b-a)/eps)) + 1)
decimal_places = 7

bisection_iteration_v1(f, a, b, max_iterations, decimal_places, type="yes")

 n         a_n         b_n         x_n       f(a_n)      f(b_n)       f(x_n)  delta=(b-a)/2^(n+1)
 1 3.000000000 3.200000000 3.100000000 -0.068403540 0.029638557 -0.020583043          0.100000000
 2 3.100000000 3.200000000 3.150000000 -0.020583043 0.029638557  0.004212533          0.050000000
 3 3.100000000 3.150000000 3.125000000 -0.020583043 0.004212533 -0.008262102          0.025000000
 4 3.125000000 3.150000000 3.137500000 -0.008262102 0.004212533 -0.002044236          0.012500000
 5 3.137500000 3.150000000 3.143750000 -0.002044236 0.004212533  0.001079255          0.006250000
 6 3.137500000 3.143750000 3.140625000 -0.002044236 0.001079255 -0.000483710          0.003125000
 7 3.140625000 3.143750000 3.142187500 -0.000483710 0.001079255  0.000297467          0.001562500
 8 3.140625000 3.142187500 3.141406250 -0.000483710 0.000297467 -0.000093197          0.000781250
 9 3.141406250 3.142187500 3.141796875 -0.000093197 0.000297467  0.000102116          0.000390625
10 3.141406250 3.141

## 5.2. Posteriori Estimation with absolute error:

- Error estimation at each iteartion: $\Delta{x_n} = |x_n - x_{n-1}|$

- If no error tolerance is given, you have to add rounded error to the final $\Delta{x_n}$.

In [11]:
def bisection_iteration_v2(f, a, b, max_iterations, decimal_places):
    # Error function
    if (f(a) * f(b) >= 0):
        print("You have not assumed right a and b\n")
        return

    # Implementing Bisection Method
    a_n = a; b_n = b; x_n = 0; x_n_minus_1 = 0; delta_x = 0;

    results = []
    for i in range(1, max_iterations + 1):
        # Find next value of x
        x_n = a_n + (b_n - a_n) / 2.0
        delta_x = abs(x_n - x_n_minus_1)

        results.append({
            'n': i,
            'a_n': a_n,
            'b_n': b_n,
            'x_n': x_n,
            'f(a_n)': f(a_n),
            'f(b_n)': f(b_n),
            'f(x_n)': f(x_n),
            'delta_x=|x_n-x_(n-1)|': delta_x
        })

        # Prepare for next iteration
        x_n_minus_1 = x_n
        if f(x_n) == 0:
            break
        elif (f(a_n) * f(x_n) < 0):
            b_n = x_n
        elif (f(a_n) * f(x_n) > 0):
            a_n = x_n

    # Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))

    if decimal_places == None:
        print(f"The value of root is: {x_n}")
    else:
        total_delta = delta_x + 0.5 * 10**(-decimal_places) #must calculate roundoff error
        print(f"The value of root with {decimal_places} decimal point is: {round(x_n, decimal_places)}")
        print(f"Relative error is: {total_delta}")

In [12]:
f = lambda x: np.e**x - np.cos(2*x)

a = -3
b = -2

max_iterations = 30
decimal_places = 5

bisection_iteration_v2(f, a, b, max_iterations, decimal_places)

 n          a_n          b_n          x_n           f(a_n)      f(b_n)           f(x_n)  delta_x=|x_n-x_(n-1)|
 1 -3.000000000 -2.000000000 -2.500000000 -9.103832183e-01 0.788978904 -2.015771868e-01        2.500000000e+00
 2 -2.500000000 -2.000000000 -2.250000000 -2.015771868e-01 0.788978904  3.161950240e-01        2.500000000e-01
 3 -2.500000000 -2.250000000 -2.375000000 -2.015771868e-01 0.316195024  5.541233632e-02        1.250000000e-01
 4 -2.500000000 -2.375000000 -2.437500000 -2.015771868e-01 0.055412336 -7.451630422e-02        6.250000000e-02
 5 -2.437500000 -2.375000000 -2.406250000 -7.451630422e-02 0.055412336 -9.791146780e-03        3.125000000e-02
 6 -2.406250000 -2.375000000 -2.390625000 -9.791146780e-03 0.055412336  2.276582202e-02        1.562500000e-02
 7 -2.406250000 -2.390625000 -2.398437500 -9.791146780e-03 0.022765822  6.474264026e-03        7.812500000e-03
 8 -2.406250000 -2.398437500 -2.402343750 -9.791146780e-03 0.006474264 -1.661944596e-03        3.906250000e-03
 

- If you have a target error tolerance $|x_n - x^*| \leq \epsilon$

    Using $|x_n - x^*| \leq |x_n - x_{n-1}| = \Delta{x_n}$ - stopping condition is now $\Delta{x_n} \leq \epsilon$

In [ ]:
def bisection_recursion_absolute(f, a, b, eps):
    # Error function
    if (f(a) * f(b) >= 0):
        print("You have not assumed right a and b\n")
        return

    # Implementing Bisection Method
    a_n = a; b_n = b; x_n = 0; x_n_minus_1 = 0;
    print(f"epsilon = {eps}")

    i = 1; results = []
    while True:
        # Find next value of x
        x_n = a_n + (b_n - a_n) / 2.0
        current_eps = abs(x_n - x_n_minus_1)

        results.append({
            'n': i,
            'a_n': a_n,
            'b_n': b_n,
            'x_n': x_n,
            'f(a_n)': f(a_n),
            'f(b_n)': f(b_n),
            'f(x_n)': f(x_n),
            'epsilon=|x_n-x_(n-1)|': current_eps
        })

        # Prepare for next iteration
        x_n_minus_1 = x_n
        if f(x_n) == 0:
            break
        elif (f(a_n) * f(x_n) < 0):
            b_n = x_n
        elif (f(a_n) * f(x_n) > 0):
            a_n = x_n

        # Stopping condition
        if current_eps <= eps:
            break
        else:
            i += 1

    # Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))

    print(f"The value of root with absolute error {eps} is: {x_n}")

In [ ]:
f = lambda x: 3*np.sin(x) + x**3 - 8*x**2 + 8*x + 1

a = 6
b = 7

eps = 0.5 * pow(10, -7) #accurate to 7 trusted decimal places

bisection_recursion_absolute (f, a, b, eps)

epsilon = 5e-08
 n         a_n         b_n         x_n        f(a_n)      f(b_n)       f(x_n)  epsilon=|x_n-x_(n-1)|
 1 6.000000000 7.000000000 6.500000000 -23.838246495 9.970959796 -9.729640036            6.500000000
 2 6.500000000 7.000000000 6.750000000  -9.729640036 9.970959796 -0.602992779            0.250000000
 3 6.750000000 7.000000000 6.875000000  -0.602992779 9.970959796  4.499775899            0.125000000
 4 6.750000000 6.875000000 6.812500000  -0.602992779 4.499775899  1.902765257            0.062500000
 5 6.750000000 6.812500000 6.781250000  -0.602992779 1.902765257  0.638531533            0.031250000
 6 6.750000000 6.781250000 6.765625000  -0.602992779 0.638531533  0.014937108            0.015625000
 7 6.750000000 6.765625000 6.757812500  -0.602992779 0.014937108 -0.294735107            0.007812500
 8 6.757812500 6.765625000 6.761718750  -0.294735107 0.014937108 -0.140075917            0.003906250
 9 6.761718750 6.765625000 6.763671875  -0.140075917 0.014937108 -0.0626136

In [ ]:
f = lambda x: x**5-17 #approximate 5th root of 17

a = 1
b = 2

eps = 0.5 * pow(10, -6) #accurate to 6 trusted decimal places

bisection_recursion_absolute (f, a, b, eps)

epsilon = 5e-07
 n         a_n         b_n         x_n        f(a_n)       f(b_n)       f(x_n)  epsilon=|x_n-x_(n-1)|
 1 1.000000000 2.000000000 1.500000000 -16.000000000 15.000000000 -9.406250000            1.500000000
 2 1.500000000 2.000000000 1.750000000  -9.406250000 15.000000000 -0.586914062            0.250000000
 3 1.750000000 2.000000000 1.875000000  -0.586914062 15.000000000  6.174285889            0.125000000
 4 1.750000000 1.875000000 1.812500000  -0.586914062  6.174285889  2.560956001            0.062500000
 5 1.750000000 1.812500000 1.781250000  -0.586914062  2.560956001  0.931820661            0.031250000
 6 1.750000000 1.781250000 1.765625000  -0.586914062  0.931820661  0.159014747            0.015625000
 7 1.750000000 1.765625000 1.757812500  -0.586914062  0.159014747 -0.217264798            0.007812500
 8 1.757812500 1.765625000 1.761718750  -0.217264798  0.159014747 -0.029959342            0.003906250
 9 1.761718750 1.765625000 1.763671875  -0.029959342  0.159014747 

## 5.3. Posteriori Estimation with relative error:

- Error estimation at each iteration: $\delta{x_n} = \dfrac{|x_n - x_{n-1}|}{|x_n|}$

- If you have a target error tolerance: $\dfrac{|x_n - x^*|}{|x_n|} \leq \eta$ 

    Using $\dfrac{|x_n - x^*|}{|x_n|} \leq \dfrac{|x_n - x_{n-1}|}{|x_n|} = \delta{x_n}$ - stopping condition is now $\delta{x_n} \leq \eta$.

In [16]:
def bisection_recursion_relative(f, a, b, eta):
    # Error function
    if (f(a) * f(b) >= 0):
        print("You have not assumed right a and b\n")
        return

    # Implementing Bisection Method
    a_n = a; b_n = b; x_n = 0; x_n_minus_1 = 0;
    print(f"eta = {eta}")

    i = 1; results = []
    while True:
        # Find next value of x
        x_n = a_n + (b_n - a_n) / 2.0
        current_eta = abs(x_n - x_n_minus_1) / abs(x_n)

        results.append({
            'n': i,
            'a_n': a_n,
            'b_n': b_n,
            'x_n': x_n,
            'f(a_n)': f(a_n),
            'f(b_n)': f(b_n),
            'f(x_n)': f(x_n),
            'eta=|x_n-x_(n-1)|/|x_n|': current_eta
        })

        # Prepare for next iteration
        x_n_minus_1 = x_n
        if f(x_n) == 0:
            break
        elif (f(a_n) * f(x_n) < 0):
            b_n = x_n
        elif (f(a_n) * f(x_n) > 0):
            a_n = x_n

        # Stopping condition
        if current_eta <= eta:
            break
        else:
            i += 1

    # Print the final result
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))

    print(f"The value of root with relative error {eta} is: {x_n}")

In [17]:
f = lambda x: np.exp(x) - np.cos(2*x)

a = -3
b = -2

eta = 0.5 * pow(10, -6)

bisection_recursion_relative (f, a, b, eta)

eta = 5e-07
 n          a_n          b_n          x_n       f(a_n)      f(b_n)       f(x_n)  eta=|x_n-x_(n-1)|/|x_n|
 1 -3.000000000 -2.000000000 -2.500000000 -0.910383218 0.788978904 -0.201577187              1.000000000
 2 -2.500000000 -2.000000000 -2.250000000 -0.201577187 0.788978904  0.316195024              0.111111111
 3 -2.500000000 -2.250000000 -2.375000000 -0.201577187 0.316195024  0.055412336              0.052631579
 4 -2.500000000 -2.375000000 -2.437500000 -0.201577187 0.055412336 -0.074516304              0.025641026
 5 -2.437500000 -2.375000000 -2.406250000 -0.074516304 0.055412336 -0.009791147              0.012987013
 6 -2.406250000 -2.375000000 -2.390625000 -0.009791147 0.055412336  0.022765822              0.006535948
 7 -2.406250000 -2.390625000 -2.398437500 -0.009791147 0.022765822  0.006474264              0.003257329
 8 -2.406250000 -2.398437500 -2.402343750 -0.009791147 0.006474264 -0.001661945              0.001626016
 9 -2.402343750 -2.398437500 -2.400390625 -

In [18]:
f = lambda x: x**5 - 3*x**3 + 2*x**2 - x + 5

a = -3
b = -2

eta = 0.05 * pow(10, -2)

bisection_recursion_relative (f, a, b, eta)

eta = 0.0005
 n      a_n          b_n          x_n         f(a_n)      f(b_n)        f(x_n)  eta=|x_n-x_(n-1)|/|x_n|
 1 -3.00000 -2.000000000 -2.500000000 -136.000000000 7.000000000 -30.781250000              1.000000000
 2 -2.50000 -2.000000000 -2.250000000  -30.781250000 7.000000000  -6.118164062              0.111111111
 3 -2.25000 -2.000000000 -2.125000000   -6.118164062 7.000000000   1.612762451              0.058823529
 4 -2.25000 -2.125000000 -2.187500000   -6.118164062 1.612762451  -1.928362846              0.028571429
 5 -2.18750 -2.125000000 -2.156250000   -1.928362846 1.612762451  -0.080791146              0.014492754
 6 -2.15625 -2.125000000 -2.140625000   -0.080791146 1.612762451   0.784742079              0.007299270
 7 -2.15625 -2.140625000 -2.148437500   -0.080791146 0.784742079   0.356725952              0.003636364
 8 -2.15625 -2.148437500 -2.152343750   -0.080791146 0.356725952   0.139162749              0.001814882
 9 -2.15625 -2.152343750 -2.154296875   -0.08079114